# Code_4 — Stronger Baselines and the Retrieval Experiment

**GPU: yes** (LLM baselines). Run after Code_3.

**Part 1 — baselines (#8)**, all on the same synthetic test calls as the schema-only system:

| Baseline | Detects | Corrects | Needs reference? |
|---|---|---|---|
| Reference metrics from Code_2 (EM, F1, NLI, BERTScore) | yes | no | **yes** |
| Standard JSON Schema validation (`jsonschema` library) | yes | no | no |
| Similarity-only correction (nearest function by embedding + nearest parameter name) | via JSON Schema | yes | no |
| LLM as verifier (asks the LLM whether the call is valid and which error) | yes | no | no |
| LLM as corrector (asks the LLM to repair the call) | — | yes | no |
| Constrained decoding | generation-time method → evaluated on real calls in **Code_5** | | |

Recent published tool-hallucination detectors can be added through `literature_baseline()` once
a method with released code is chosen (see the note in that cell).

**Part 2 — retrieval (#7)**, presented as an *exploratory* component: the calls that the schema-only
system flagged for an unresolved value are sent to a web search, candidate values are extracted from
the snippets and verified with an NLI model. Everything about it is logged and explained below.

Outputs: `runs/<run_id>/baselines.json`, `runs/<run_id>/retrieval.json`; cache in `cache/`.

In [ ]:
import os, sys, json, importlib, subprocess, time, copy, random, re
from collections import Counter, defaultdict

IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/hallucination_project_v2'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers', 'rapidfuzz', 'ddgs', 'jsonschema'],
                   check=True)
else:
    DRIVE_ROOT = os.environ.get('TOOLHALL_ROOT', os.path.abspath('hallucination_project_v2'))
CODE_DIR = os.path.join(DRIVE_ROOT, 'code')
sys.path.insert(0, CODE_DIR)
import toolhall_core as th
importlib.reload(th)
import numpy as np

NOTEBOOK = 'Code_4'
CFG = th.load_config(os.path.join(CODE_DIR, 'config.json'))
CFG['drive_root'] = DRIVE_ROOT
P = th.Paths(CFG)
SEED = CFG['seed']
th.set_seed(SEED)
NB, ALPHA = CFG['bootstrap']['n'], CFG['bootstrap']['alpha']
RUN_LLM_BASELINES = True
RUN_RETRIEVAL = True
QUICK_TEST = None

raw_registry = th.load_json(P.data_file('registry.json'), CFG, kind='data')
single = th.load_json(P.data_file('benchmark_single.json'), CFG, kind='data')
splits = th.load_json(P.data_file('splits.json'), CFG, kind='data')
core = th.load_json(P.run_file('core_results.json'), CFG)
core_records = th.load_json(P.run_file('core_records.json'), CFG)
refm = th.load_json(P.run_file('reference_metrics.json'), CFG)
REG = th.Registry(raw_registry, CFG['validator']['unknown_required_policy'])
BY_ID = {ex['id']: ex for ex in single}
MATCHER = th.FunctionMatcher(REG, model_name=CFG['matcher']['model_name'], cache_dir=P.cache)
print(f"run_id={CFG['run_id']} | tuned config from Code_3: {core['tuned']}")

## Shared scoring helpers

In [ ]:
def det_summary(y_true, y_pred, true_types=None, pred_primary=None):
    b = th.binary_metrics(y_true, y_pred)
    out = {'binary': b, 'ci_balanced_accuracy': th.bootstrap_ci(
        lambda a, c: th.binary_metrics(a, c)['balanced_accuracy'], y_true, y_pred, n=NB, alpha=ALPHA, seed=SEED)}
    if true_types is not None and pred_primary is not None:
        out['classification'] = th.multiclass_metrics(true_types, pred_primary,
                                                      labels=th.ALL_LABELS + ['UNSPECIFIED'])
    return out


def test_ids(split_name):
    return [single[i]['id'] for i in splits[split_name]['test']]


SYSTEM = {s: {r['id']: r for r in core_records[s]} for s in ('random', 'function_disjoint')}

## Baseline 1 — standard JSON Schema validation

Each function's schema is converted to a JSON Schema: envelope must be exactly
`{function, parameters}`; `parameters` has the declared properties with their JSON types and enums,
`required` from the registry, and `additionalProperties: false`. Validation errors are mapped to the
taxonomy (unknown function → WRONG_FUNCTION, additionalProperties → EXTRA, required → MISSING,
type → WRONG_TYPE, enum/minimum/maximum → WRONG_VALUE, envelope → WRONG_FORMAT).
This is plain, standard validation: no type calibration, no empty-value convention, no correction.

In [ ]:
import jsonschema

JSON_TYPE = {'string': 'string', 'integer': 'integer', 'number': 'number', 'boolean': 'boolean',
             'array': 'array', 'object': 'object'}
_schema_cache = {}


def json_schema_for(fn):
    if fn in _schema_cache:
        return _schema_cache[fn]
    props, req = {}, []
    for p, s in REG.params(fn).items():
        d = {}
        if s.get('type') in JSON_TYPE:
            d['type'] = JSON_TYPE[s['type']]
        if s.get('enum'):
            d['enum'] = s['enum']
        for k in ('minimum', 'maximum'):
            if isinstance(s.get(k), (int, float)):
                d[k] = s[k]
        props[p] = d
        if REG.is_required(s):
            req.append(p)
    sch = {'type': 'object', 'additionalProperties': False, 'required': ['function', 'parameters'],
           'properties': {'function': {'type': 'string'},
                          'parameters': {'type': 'object', 'properties': props, 'required': req,
                                         'additionalProperties': False}}}
    _schema_cache[fn] = jsonschema.Draft7Validator(sch)
    return _schema_cache[fn]


def jsonschema_validate(raw):
    d = th.parse_raw_call(raw)
    if d is None:
        return ['WRONG_FORMAT']
    types = set()
    if set(d.keys()) != {'function', 'parameters'}:
        types.add('WRONG_FORMAT')
    fn_name = d.get('function') if isinstance(d.get('function'), str) else None
    fn = REG.resolve(fn_name) if fn_name else None
    if fn is None:
        types.add('WRONG_FUNCTION' if fn_name else 'WRONG_FORMAT')
        return sorted(types, key=th.PRIMARY_ORDER.index)
    for err in json_schema_for(fn).iter_errors(d):
        path = list(err.absolute_path)
        if not path:
            types.add('WRONG_FORMAT')
        elif err.validator == 'additionalProperties':
            types.add('EXTRA_PARAMETER')
        elif err.validator == 'required':
            types.add('MISSING_PARAMETER')
        elif err.validator == 'type':
            types.add('WRONG_TYPE' if len(path) >= 2 else 'WRONG_FORMAT')
        elif err.validator in ('enum', 'minimum', 'maximum'):
            types.add('WRONG_VALUE')
        else:
            types.add('WRONG_FORMAT')
    return sorted(types, key=th.PRIMARY_ORDER.index)


BASE = {}
JS_PRED = {}
for split_name in ['random', 'function_disjoint']:
    exs = [BY_ID[i] for i in test_ids(split_name)]
    preds = [jsonschema_validate(ex['generated_tool_call']) for ex in exs]
    JS_PRED[split_name] = {ex['id']: p for ex, p in zip(exs, preds)}
    y = [int(ex['is_hallucinated']) for ex in exs]
    tt = [ex['error_types'][0] if ex['error_types'] else 'NONE' for ex in exs]
    BASE.setdefault(split_name, {})['json_schema_validation'] = det_summary(
        y, [int(bool(p)) for p in preds], tt, [p[0] if p else 'NONE' for p in preds])
    b = BASE[split_name]['json_schema_validation']['binary']
    print(f"[{split_name}] JSON Schema validation: bal.acc {b['balanced_accuracy']:.3f}  F1 {b['f1']:.3f}  "
          f"FPR {b['fpr']:.3f}  FNR {b['fnr']:.3f}")

## Baseline 2 — similarity-only correction

Detection: the JSON Schema validator above. Correction, using similarity only: envelope rewritten,
function name replaced by the registry function with the highest embedding similarity (no threshold),
each undeclared parameter name replaced by the most similar declared name (edit similarity ≥ 0.5) or
dropped. No type casting, no filling of missing parameters.

In [ ]:
sim_matcher = MATCHER.configured(use_semantic=MATCHER.use_semantic, use_edit=not MATCHER.use_semantic,
                                 w_sem=1.0, min_score=-1.0)


def similarity_correct(raw):
    n = th.normalize_tool_call(raw)
    if n is None:
        return None
    fn = REG.resolve(n['function']) or sim_matcher.match(n['function']) or n['function']
    schema = REG.params(fn)
    params = {}
    for p, v in n['parameters'].items():
        if p in schema:
            params[p] = v
            continue
        cands = [(1 - th.edit_norm(p, q), q) for q in schema if q not in n['parameters']]
        if cands:
            sc, q = max(cands)
            if sc >= 0.5 and q not in params:
                params[q] = v
    return {'function': fn, 'parameters': params}


for split_name in ['random', 'function_disjoint']:
    exs = [BY_ID[i] for i in test_ids(split_name)]
    exact = []
    for ex in exs:
        flagged = bool(JS_PRED[split_name][ex['id']])
        final = similarity_correct(ex['generated_tool_call']) if flagged else th.parse_raw_call(ex['generated_tool_call'])
        exact.append(int(th.exact_strict(final, ex['ground_truth_tool_call'])))
    sys_exact = [int(SYSTEM[split_name][ex['id']]['final_exact']) for ex in exs]
    BASE[split_name]['similarity_correction'] = {
        'exact_all': float(np.mean(exact)),
        'exact_hallucinated': float(np.mean([e for e, ex in zip(exact, exs) if ex['is_hallucinated']])),
        'vs_schema_only': {'mcnemar': th.mcnemar_exact(sys_exact, exact),
                           'diff_ci': th.paired_bootstrap_diff(sys_exact, exact, n=NB, alpha=ALPHA, seed=SEED)}}
    print(f"[{split_name}] similarity-only correction exact {np.mean(exact):.3f} vs schema-only {np.mean(sys_exact):.3f}"
          f"  (McNemar p={BASE[split_name]['similarity_correction']['vs_schema_only']['mcnemar']['p_value']:.2g})")

## Baselines 3 and 4 — LLM as verifier and LLM as corrector

Same information as the schema-only system: the user request, the broken call, and the schema of the
named function (or, if the name is not in the registry, the schemas of the 5 most similar registry
functions). Chat template, greedy decoding. Run on a stratified subset of each test split
(`models.llm_baseline_limit`), and the schema-only system is scored on the **same** calls for the
paired McNemar test.

In [ ]:
TYPES_TXT = ', '.join(th.ERROR_TYPES)


def schema_context(raw):
    n = th.normalize_tool_call(raw)
    name = n['function'] if n else ''
    fn = REG.resolve(name)
    if fn:
        return [th.schema_for_prompt(REG, fn)]
    cands = MATCHER.configured(min_score=-1.0).score_candidates(name) if name else []
    top = [c for c, _ in sorted(cands, key=lambda x: -x[1])[:5]]
    return [th.schema_for_prompt(REG, c) for c in top]


def verifier_messages(ex):
    return [{'role': 'system', 'content': 'You check API tool calls against their schema. Answer with JSON only.'},
            {'role': 'user', 'content':
                f"User request: {ex['query'][:600]}\n"
                f"Available function schema(s): {json.dumps(schema_context(ex['generated_tool_call']))[:2500]}\n"
                f"Tool call to check: {json.dumps(th.parse_raw_call(ex['generated_tool_call']), default=str)[:800]}\n"
                f"The expected envelope is {{\"function\": ..., \"parameters\": {{...}}}}.\n"
                f"Is the tool call correct? Reply exactly as JSON: "
                f"{{\"valid\": true or false, \"error_types\": [zero or more of: {TYPES_TXT}]}}"}]


def corrector_messages(ex):
    return [{'role': 'system', 'content': 'You repair malformed API tool calls. Answer with JSON only.'},
            {'role': 'user', 'content':
                f"User request: {ex['query'][:600]}\n"
                f"Available function schema(s): {json.dumps(schema_context(ex['generated_tool_call']))[:2500]}\n"
                f"Broken tool call: {json.dumps(th.parse_raw_call(ex['generated_tool_call']), default=str)[:800]}\n"
                f"Reply with ONLY the corrected call as JSON: {{\"function\": \"...\", \"parameters\": {{...}}}}"}]


def parse_verdict(text):
    d = th.parse_raw_call(text) or {}
    valid = d.get('valid')
    if isinstance(valid, str):
        valid = valid.strip().lower() == 'true'
    types = [t for t in (d.get('error_types') or []) if t in th.ERROR_TYPES] if isinstance(d.get('error_types'), list) else []
    if valid is None:
        low = text.lower()
        valid = ('"valid": true' in low) or (low.strip().startswith('valid') and 'invalid' not in low)
        parsed = False
    else:
        parsed = True
    return bool(valid), sorted(set(types), key=th.PRIMARY_ORDER.index), parsed


def llm_subset(split_name, limit):
    ids = test_ids(split_name)
    by = defaultdict(list)
    for i in ids:
        by[BY_ID[i]['hallucination_type']].append(i)
    per = max(1, limit // len(by))
    pick = []
    for k in sorted(by):
        L = sorted(by[k])
        random.Random(SEED).shuffle(L)
        pick += L[:per]
    return sorted(pick)


LLM_BASE = {}
if RUN_LLM_BASELINES:
    name = CFG['models']['llm_baseline']
    tok, model, device = th.load_causal_lm(name)
    print(f"loaded {name} on {device}")
    cache_path = P.cache_file(f"llm_baselines_{name.replace('/', '_')}.jsonl")
    done = {(r['id'], r['task']): r for r in th.read_jsonl(cache_path)}
    for split_name in ['random', 'function_disjoint']:
        ids = llm_subset(split_name, CFG['models']['llm_baseline_limit'])
        if QUICK_TEST:
            ids = ids[::max(1, len(ids) // QUICK_TEST)][:QUICK_TEST]
        for task, build in (('verify', verifier_messages), ('correct', corrector_messages)):
            todo = [i for i in ids if (i, task) not in done]
            print(f"[{split_name}] {task}: {len(ids)} calls, {len(todo)} to generate")
            for start in range(0, len(todo), 32):
                chunk = todo[start:start + 32]
                prompts = [th.chat_text(tok, build(BY_ID[i])) for i in chunk]
                texts, secs = th.generate_batch(tok, model, device, prompts, max_new_tokens=160, batch_size=8)
                for i, t, s in zip(chunk, texts, secs):
                    rec = {'id': i, 'task': task, 'output': t, 'seconds': s}
                    th.append_jsonl(cache_path, rec)
                    done[(i, task)] = rec
        exs = [BY_ID[i] for i in ids]
        y = [int(ex['is_hallucinated']) for ex in exs]
        verd = [parse_verdict(done[(i, 'verify')]['output']) for i in ids]
        pred_flag = [int(not v[0]) for v in verd]
        pred_primary = [(v[1][0] if v[1] else ('NONE' if v[0] else 'UNSPECIFIED')) for v in verd]
        tt = [ex['error_types'][0] if ex['error_types'] else 'NONE' for ex in exs]
        sys_flag = [int(SYSTEM[split_name][i]['flagged']) for i in ids]
        corr_exact = []
        for i in ids:
            fixed = th.parse_raw_call(done[(i, 'correct')]['output'])
            corr_exact.append(int(th.exact_strict(fixed, BY_ID[i]['ground_truth_tool_call'])))
        sys_exact = [int(SYSTEM[split_name][i]['final_exact']) for i in ids]
        sys_det_ok = [int(f == t) for f, t in zip(sys_flag, y)]
        llm_det_ok = [int(f == t) for f, t in zip(pred_flag, y)]
        LLM_BASE[split_name] = {
            'model': name, 'n': len(ids),
            'verifier': {**det_summary(y, pred_flag, tt, pred_primary),
                         'parse_rate': float(np.mean([v[2] for v in verd])),
                         'vs_schema_only_detection': th.mcnemar_exact(sys_det_ok, llm_det_ok),
                         'mean_seconds': float(np.mean([done[(i, 'verify')]['seconds'] for i in ids]))},
            'corrector': {'exact_all': float(np.mean(corr_exact)),
                          'exact_hallucinated': float(np.mean([e for e, ex in zip(corr_exact, exs) if ex['is_hallucinated']])),
                          'schema_only_exact_same_subset': float(np.mean(sys_exact)),
                          'vs_schema_only': {'mcnemar': th.mcnemar_exact(sys_exact, corr_exact),
                                             'diff_ci': th.paired_bootstrap_diff(sys_exact, corr_exact, n=NB,
                                                                                 alpha=ALPHA, seed=SEED)},
                          'mean_seconds': float(np.mean([done[(i, 'correct')]['seconds'] for i in ids]))},
            'schema_only_same_subset': det_summary(y, sys_flag)}
        v, c = LLM_BASE[split_name]['verifier'], LLM_BASE[split_name]['corrector']
        print(f"[{split_name}] LLM verifier bal.acc {v['binary']['balanced_accuracy']:.3f} "
              f"(schema-only on same calls {LLM_BASE[split_name]['schema_only_same_subset']['binary']['balanced_accuracy']:.3f}),"
              f" macro-F1 {v['classification']['macro_f1']:.3f}, parse rate {v['parse_rate']:.2f}")
        print(f"[{split_name}] LLM corrector exact {c['exact_all']:.3f} vs schema-only {c['schema_only_exact_same_subset']:.3f}"
              f"  McNemar p={c['vs_schema_only']['mcnemar']['p_value']:.2g}  ({c['mean_seconds']:.2f}s/call)")
    del model
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

## Baseline 5 — a published tool-hallucination detector (placeholder)

Fill this in once the related-work search picks a method that has released code and works on a
single tool call. It must return, per call, a flag (and optionally error types) without using the
reference call, so it is scored exactly like the rows above. Left empty on purpose: reporting a
re-implementation from a paper description alone would not be a fair comparison.

In [ ]:
def literature_baseline(examples):
    """Return a list of (flagged: bool, error_types: list) for each example, or None if not set up."""
    return None

LIT = {}
for split_name in ['random', 'function_disjoint']:
    exs = [BY_ID[i] for i in test_ids(split_name)]
    out = literature_baseline(exs)
    if out is not None:
        y = [int(ex['is_hallucinated']) for ex in exs]
        LIT[split_name] = det_summary(y, [int(f) for f, _ in out])
print('literature baseline:', 'not configured' if not LIT else 'done')

## Comparison tables — detection and correction kept separate

In [ ]:
COMPARE = {}
for split_name in ['random', 'function_disjoint']:
    th.print_header(f"DETECTION COMPARISON — {split_name} test split")
    print(f"{'method':<34}{'deployable':>11}{'bal.acc':>9}{'F1':>7}{'FPR':>7}{'FNR':>7}{'macroF1':>9}")
    rows = []
    for name, rep in refm['results'][split_name].items():
        b = rep['binary']
        rows.append((f"{name}", 'no', b['balanced_accuracy'], b['f1'], b['fpr'], b['fnr'], None))
    js = BASE[split_name]['json_schema_validation']
    rows.append(('JSON Schema validation', 'yes', js['binary']['balanced_accuracy'], js['binary']['f1'],
                 js['binary']['fpr'], js['binary']['fnr'], js['classification']['macro_f1']))
    m = core['main'][split_name]['detection']
    rows.append(('schema-only (ours)', 'yes', m['binary']['balanced_accuracy'], m['binary']['f1'],
                 m['binary']['fpr'], m['binary']['fnr'], m['classification']['macro_f1']))
    if split_name in LLM_BASE:
        v = LLM_BASE[split_name]['verifier']
        rows.append((f"LLM verifier (subset n={LLM_BASE[split_name]['n']})", 'yes', v['binary']['balanced_accuracy'],
                     v['binary']['f1'], v['binary']['fpr'], v['binary']['fnr'], v['classification']['macro_f1']))
        s = LLM_BASE[split_name]['schema_only_same_subset']['binary']
        rows.append(('schema-only on that subset', 'yes', s['balanced_accuracy'], s['f1'], s['fpr'], s['fnr'], None))
    for r in rows:
        print(f"{r[0]:<34}{r[1]:>11}" + ''.join(f"{x:>9.3f}" if isinstance(x, float) else f"{'—':>9}" for x in r[2:]))
    th.print_header(f"CORRECTION COMPARISON (exact match with reference) — {split_name} test split")
    c = core['main'][split_name]['correction']
    abl = core['ablation'][split_name]
    crow = [('no correction (block flagged calls)', abl['detect_only (no correction)']['summary']['exact_all']),
            ('similarity-only correction', BASE[split_name]['similarity_correction']['exact_all']),
            ('schema-only (ours)', c['final_exact_rate_all'])]
    if split_name in LLM_BASE:
        crow += [(f"LLM corrector (subset)", LLM_BASE[split_name]['corrector']['exact_all']),
                 ('schema-only on that subset', LLM_BASE[split_name]['corrector']['schema_only_exact_same_subset'])]
    for n_, v_ in crow:
        print(f"{n_:<40}{v_:>8.3f}")
    COMPARE[split_name] = {'detection_rows': rows, 'correction_rows': crow}

## Part 2 — Retrieval for unresolved values (exploratory, #7)

How it works, exactly:
* **Which calls** — the calls whose remaining problem is a *value* (a placeholder the corrector had
  to invent for a missing parameter, or a value flagged as ungrounded); taken from the system's own
  flags in Code_3, not from the ground truth.
* **Retrieval query** — two variants, both logged: `schema_names` = "<function> API <parameter> valid values"
  (the previous version's query), and `user_query` = "<first 120 characters of the user request> <parameter>".
* **What is searched / stored** — there is no local database: DuckDuckGo text search (`ddgs`), the top
  `retrieval.max_results` result snippets. Every query and its snippets are cached in
  `cache/retrieval_cache.json`, so re-runs are deterministic and offline.
* **Model and similarity** — no embedding retrieval. Candidate values are pulled from the snippets with
  regular expressions (numbers, dates, quoted strings, `param=value`) near the parameter name; each
  candidate is verified with the NLI cross-encoder (premise = snippet, hypothesis = "The <param> is <value>.")
  using the **entailment probability** (column found by label name).
* **Threshold** — the best candidate is used only if its entailment probability ≥ `retrieval.nli_entail_threshold`.
* **Scoring** — the candidate is inserted and the WHOLE call must then equal the reference
  (the previous version credited a call if any one parameter matched, even if the rest was wrong).
  "Reference value among candidates" is the retrieval recall upper bound.

In [ ]:
RETR = {}
if RUN_RETRIEVAL:
    try:
        from ddgs import DDGS
    except ImportError:
        from duckduckgo_search import DDGS
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    nli_tok = AutoTokenizer.from_pretrained(CFG['models']['nli_model'])
    nli_model = AutoModelForSequenceClassification.from_pretrained(CFG['models']['nli_model']).to(DEVICE).eval()
    ENT = next(int(i) for i, l in nli_model.config.id2label.items() if 'entail' in l.lower())
    cache_file = P.cache_file('retrieval_cache.json')
    rcache = json.load(open(cache_file, encoding='utf-8')) if os.path.exists(cache_file) else {}
    n_network = 0

    def search(q):
        global n_network
        if q in rcache:
            return rcache[q]
        snippets = []
        try:
            with DDGS() as d:
                for r in d.text(q, max_results=CFG['retrieval']['max_results']):
                    body = (r.get('body') or '').strip()
                    if len(body) > 20:
                        snippets.append(body)
        except Exception as e:
            snippets = []
        rcache[q] = snippets
        n_network += 1
        time.sleep(CFG['retrieval']['sleep_seconds'])
        if n_network % 25 == 0:
            json.dump(rcache, open(cache_file, 'w', encoding='utf-8'))
        return snippets

    def candidates(snips, param):
        out = []
        pn = param.replace('_', ' ').lower()
        for s in snips:
            low = s.lower()
            pos = low.find(pn)
            if pos == -1:
                pos = low.find(param.lower())
            win = s[max(0, pos - 100): pos + 300] if pos != -1 else s[:300]
            out += re.findall(r'\b\d{4}-\d{2}-\d{2}\b', win)
            out += re.findall(r'\b\d+(?:\.\d+)?\b', win)
            out += [q for q in re.findall(r'["\']([^"\']{2,50})["\']', win)
                    if q.lower() not in ('true', 'false', 'null', 'none')]
            out += [m.strip('"\'') for m in re.findall(rf'{re.escape(param)}\s*[=:]\s*([^\s,;{{}}]+)', s, re.I)]
        return list(dict.fromkeys(o for o in out if o))[:8]

    def entail(pairs):
        if not pairs:
            return []
        enc = nli_tok([p for p, _ in pairs], [h for _, h in pairs], return_tensors='pt', truncation=True,
                      max_length=512, padding=True).to(DEVICE)
        with torch.no_grad():
            return torch.softmax(nli_model(**enc).logits, -1)[:, ENT].cpu().tolist()

    for setting, targets_by_split in [('schema only + retrieval', '1 schema only'),
                                      ('schema + query + retrieval', '2 schema + query')]:
        RETR[setting] = {}
        for split_name in ['random']:
            targets = core_records['retrieval_targets'][split_name][targets_by_split]
            if QUICK_TEST:
                targets = targets[:QUICK_TEST]
            base_exact = core_records['wrong_value_exact_by_id'][split_name][targets_by_split]
            for qmode in CFG['retrieval']['query_modes']:
                stats = Counter()
                new_exact = dict(base_exact)
                log = []
                for t in targets:
                    ex = BY_ID[t['id']]
                    call = copy.deepcopy(t['final_call'])
                    if not isinstance(call, dict) or 'parameters' not in call:
                        continue
                    gt_params = ex['ground_truth_tool_call']['parameters']
                    for p in t['flagged_params']:
                        if p not in call['parameters']:
                            continue
                        fn = call['function']
                        q = (f"{fn.replace('.', ' ').replace('_', ' ')} API {p.replace('_', ' ')} valid values"
                             if qmode == 'schema_names' else f"{ex['query'][:120]} {p.replace('_', ' ')}")
                        snips = search(q)
                        cands = candidates(snips, p)
                        stats['params'] += 1
                        stats['with_snippets'] += int(bool(snips))
                        stats['with_candidates'] += int(bool(cands))
                        stats['reference_value_among_candidates'] += int(str(gt_params.get(p)) in [str(c) for c in cands])
                        if not cands:
                            continue
                        pairs = [(s[:512], f"The {p.replace('_', ' ')} is {c}.") for c in cands for s in snips[:2]]
                        sc = entail(pairs)
                        best = {}
                        for (pr, hy), v in zip(pairs, sc):
                            c = hy[len(f"The {p.replace('_', ' ')} is "):-1]
                            best[c] = max(best.get(c, 0.0), v)
                        c_best, v_best = max(best.items(), key=lambda x: x[1])
                        if v_best >= CFG['retrieval']['nli_entail_threshold']:
                            decl = REG.params(fn).get(p, {}).get('type')
                            val = th.safe_cast(c_best, decl) if decl else c_best
                            call['parameters'][p] = val if val is not None else c_best
                            stats['value_inserted'] += 1
                            stats['inserted_value_correct'] += int(str(call['parameters'][p]) == str(gt_params.get(p)))
                    ok = th.exact_strict(call, ex['ground_truth_tool_call'])
                    stats['calls'] += 1
                    stats['call_now_exact'] += int(ok)
                    stats['call_was_exact'] += int(t['final_exact_before'])
                    new_exact[t['id']] = bool(ok)
                    log.append({'id': t['id'], 'query_mode': qmode, 'exact_before': t['final_exact_before'],
                                'exact_after': ok})
                test_ids_ = test_ids(split_name)
                before = float(np.mean([base_exact[i] for i in test_ids_]))
                after = float(np.mean([new_exact[i] for i in test_ids_]))
                wv_ids = [i for i in test_ids_ if BY_ID[i]['error_types'] == ['WRONG_VALUE']]
                RETR[setting][f"{split_name}/{qmode}"] = {
                    'n_target_calls': len(targets), 'stats': dict(stats),
                    'end_to_end_exact_before': before, 'end_to_end_exact_after': after,
                    'wrong_value_exact_before': float(np.mean([base_exact[i] for i in wv_ids])),
                    'wrong_value_exact_after': float(np.mean([new_exact[i] for i in wv_ids])),
                    'log': log}
                print(f"{setting:<28} [{split_name}/{qmode:<12}] targets {len(targets):>4} | params {stats['params']:>4} "
                      f"| snippets {stats['with_snippets']:>4} | candidates {stats['with_candidates']:>4} "
                      f"| ref value among cands {stats['reference_value_among_candidates']:>3} "
                      f"| inserted {stats['value_inserted']:>3} (correct {stats['inserted_value_correct']:>3}) "
                      f"| end-to-end {before:.4f} -> {after:.4f}")
    json.dump(rcache, open(cache_file, 'w', encoding='utf-8'))
    RETR['_settings'] = {k: CFG['retrieval'][k] for k in CFG['retrieval']}
    RETR['_settings'].update({'search_engine': 'DuckDuckGo text search via ddgs', 'nli_model': CFG['models']['nli_model'],
                              'verification': 'entailment probability, premise=snippet, hypothesis="The <param> is <value>."',
                              'network_queries_this_run': n_network, 'cache_entries': len(rcache)})
    print('retrieval cache entries:', len(rcache), '| new network queries this run:', n_network)

## Save

In [ ]:
th.save_json({'baselines': BASE, 'llm_baselines': LLM_BASE, 'literature_baseline': LIT, 'comparison': COMPARE,
              'hardware': th.hardware_info(), 'packages': th.package_versions()},
             P.run_file('baselines.json'), CFG, NOTEBOOK)
th.save_json(RETR, P.run_file('retrieval.json'), CFG, NOTEBOOK)
print('saved ->', P.run_file('baselines.json'), '|', P.run_file('retrieval.json'))
print('Code_4 complete.')